**TÍTULO**

F3N3 — Classificação das fases do ciclo de vida: gênese, crescimento, faixa de pico e decaimento

**CONTEXTO**

No paradigma de recarga-descarga (Jin, 1997), o El Niño nasce de um estado recarregado de calor subsuperficial (gênese), amplifica-se pelo acoplamento com o vento (crescimento), atinge um patamar quase estacionário (faixa de pico) e decai quando a descarga de calor para fora do equador domina (decaimento).

**DESAFIO**

Hipótese: as quatro fases têm assinaturas temporais distintas e duração sistematicamente assimétrica (crescimento mais longo que decaimento em eventos fortes). Objetivos: segmentar cada evento do catálogo F3N2 nas quatro fases, justificar fisicamente as janelas e quantificar durações por classe.

**METODOLOGIA**

Faixa de pico: patamar contínuo com SSTA suavizada ≥ 90% do máximo do evento (documentação do projeto). Crescimento: do cruzamento de +0,5 °C até o início do patamar. Decaimento: do fim do patamar até a saída do limiar. Gênese: até 26 semanas antes do cruzamento, iniciando no último mínimo local da SSTA suavizada — janela coerente com o tempo de recarga equatorial de Jin (1997) e Meinen & McPhaden (2000).

**RESULTADOS ESPERADOS**

TabF3N3_fases_semanais.csv (fase de cada semana), TabF3N3_duracao_fases.csv (duração por evento e fase), FigF3N3_linha_do_tempo (série colorida por fase) e FigF3N3_duracao_por_classe (duração das fases por classe de evento).

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. JIN, F.-F. An Equatorial Ocean Recharge Paradigm for ENSO. Part I: Conceptual Model. Journal of the Atmospheric Sciences, v. 54, p. 811-829, 1997. DOI: https://doi.org/10.1175/1520-0469(1997)054<0811:AEORPF>2.0.CO;2
2. MEINEN, C. S.; McPHADEN, M. J. Observations of Warm Water Volume Changes in the Equatorial Pacific and Their Relationship to El Niño and La Niña. Journal of Climate, v. 13, p. 3551-3559, 2000. DOI: https://doi.org/10.1175/1520-0442(2000)013<3551:OOWWVC>2.0.CO;2
3. TRENBERTH, K. E. The Definition of El Niño. Bulletin of the American Meteorological Society, v. 78, p. 2771-2777, 1997. DOI: https://doi.org/10.1175/1520-0442(1997)010<2759:TDOENO>2.0.CO;2

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
from nino_brasil import fase3_nino as f3

f3.ensure_dirs()
master = f3.load_master_weekly()
fisicas = [c for c in master.columns if c != 'ocean_source_code']
print(f'Master semanal F2: {master.shape[0]} semanas x {len(fisicas)} variaveis fisicas'
      f" | simulado={master.attrs.get('dado_simulado', False)}")
print(f'Periodo: {master.index.min().date()} a {master.index.max().date()}')

Master semanal F2: 2376 semanas x 44 variaveis fisicas | simulado=False
Periodo: 1981-01-04 a 2026-07-12


In [2]:
ssta = pd.to_numeric(master['nino34_ssta'], errors='coerce')
suave = f3.smooth_ssta(ssta, 13)
catalogo = f3.detect_el_nino_events(ssta)
fases = f3.classify_life_cycle(ssta, catalogo, peak_fraction=0.90)
tabela_fases = pd.DataFrame({'ssta_suavizada': suave, 'fase': fases})
f3.save_table(tabela_fases, 'TabF3N3_fases_semanais')
tabela_fases['fase'].value_counts()

[tabela] TabF3N3_fases_semanais.csv (2376x2) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


fase
neutro         1663
genese          231
crescimento     199
faixa_pico      148
decaimento      135
Name: count, dtype: int64

In [3]:
linhas = []
for _, ev in catalogo.iterrows():
    bloco = fases.loc[pd.Timestamp(ev['inicio']) - pd.Timedelta(weeks=26):pd.Timestamp(ev['fim'])]
    contagem = bloco.value_counts()
    linhas.append({'evento': ev['evento'], 'classe': ev['classe'],
                   **{f'{fase}_semanas': int(contagem.get(fase, 0))
                      for fase in ('genese', 'crescimento', 'faixa_pico', 'decaimento')}})
duracoes = pd.DataFrame(linhas)
f3.save_table(duracoes, 'TabF3N3_duracao_fases', index=False)
duracoes

[tabela] TabF3N3_duracao_fases.csv (11x6) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


,evento,classe,genese_semanas,crescimento_semanas,faixa_pico_semanas,decaimento_semanas
0,EN_1983,muito_forte,26,21,12,16
1,EN_1987,fraco,26,11,8,9
2,EN_1987,moderado,1,18,18,25
3,EN_1992,forte,22,17,8,17
4,EN_1997,muito_forte,25,17,18,13
5,EN_2002,moderado,26,18,9,10
6,EN_2004,fraco,18,2,19,3
7,EN_2009,forte,26,20,10,11
8,EN_2015,muito_forte,9,53,15,14
9,EN_2018,fraco,26,6,25,11


In [4]:
cores = {'neutro': '0.85', 'genese': '#7fbf7f', 'crescimento': '#f4a742',
         'faixa_pico': '#d62728', 'decaimento': '#1f77b4'}
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(suave.index, suave, color='k', lw=0.8)
for fase, cor in cores.items():
    if fase == 'neutro':
        continue
    selecao = fases == fase
    ax.fill_between(suave.index, 0, suave.where(selecao), color=cor, alpha=0.7, label=fase)
ax.axhline(0.5, color='k', ls='--', lw=0.6)
ax.set_ylabel('SSTA suavizada (C)')
ax.set_title('Ciclo de vida do El Nino: genese, crescimento, faixa de pico (>=90% do maximo) e decaimento')
ax.legend(loc='upper left', fontsize=8, ncol=4)
fig.tight_layout()
f3.save_figure(fig, 'FigF3N3_linha_do_tempo')
plt.close(fig)

colunas_fase = ['genese_semanas', 'crescimento_semanas', 'faixa_pico_semanas', 'decaimento_semanas']
media_classe = duracoes.groupby('classe')[colunas_fase].mean().round(1)
f3.save_table(media_classe, 'FigF3N3_duracao_por_classe_dados')
fig, ax = plt.subplots(figsize=(9, 4.5))
media_classe.plot(kind='bar', ax=ax)
ax.set_ylabel('Duracao media (semanas)')
ax.set_title('Duracao media das fases por classe de evento')
fig.tight_layout()
f3.save_figure(fig, 'FigF3N3_duracao_por_classe')
plt.close(fig)

[figura] FigF3N3_linha_do_tempo.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] FigF3N3_duracao_por_classe_dados.csv (4x4) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N3_duracao_por_classe.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
